# 07 - EDA con fuentes externas

**Objetivo.** Iniciar el análisis exploratorio enriquecido con fuentes externas. En esta versión se integra Censo 2022 por comuna para construir tasas simples de oferta publicada respecto de población y superficie.

In [ ]:
from pathlib import Path
import json
import re
import unicodedata
from datetime import datetime

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option('display.max_columns', 140)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 180)


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'data' / 'raw').exists() and (path / 'README.md').exists():
            return path
    raise FileNotFoundError('No se encontro la raiz del repo. Ejecutar dentro del proyecto.')

REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / 'data' / 'raw'
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
REPORTS_DIR = PROCESSED_DIR / 'reports'
EXTERNAL_DIR = REPO_ROOT / 'Fuentes_de_enriquecimiento'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)
print('RAW_DIR:', RAW_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)

## Carga de base inmobiliaria y Censo 2022

In [ ]:
base_candidates = [
    PROCESSED_DIR / 'df_publicaciones_consolidadas_con_outliers_v1.csv',
    PROCESSED_DIR / 'df_publicaciones_consolidadas_con_flags_nulos_v1.csv',
    PROCESSED_DIR / 'df_publicaciones_consolidadas_v1.csv',
]
for path in base_candidates:
    if path.exists():
        base_path = path
        break
else:
    raise FileNotFoundError('No existe base consolidada.')

censo_path = PROCESSED_DIR / 'censo_2022_comunas_v1.csv'
if not censo_path.exists():
    raise FileNotFoundError('No existe censo_2022_comunas_v1.csv. Ejecutar notebook 06.')

df_base = pd.read_csv(base_path, low_memory=False)
censo_2022 = pd.read_csv(censo_path)
print('Base:', base_path.relative_to(REPO_ROOT), df_base.shape)
print('Censo:', censo_2022.shape)

## Agregación por comuna

In [ ]:
df_base_comuna = df_base[df_base['comuna'].notna()].copy()
df_base_comuna['comuna'] = df_base_comuna['comuna'].astype(int)

oferta_comuna = (
    df_base_comuna
    .groupby(['comuna', 'tipo_operacion_norm'])
    .size()
    .reset_index(name='publicaciones')
)

total_comuna = df_base_comuna.groupby('comuna').size().reset_index(name='publicaciones_totales')
base_enriquecida_comuna = total_comuna.merge(censo_2022, on='comuna', how='left')
base_enriquecida_comuna['publicaciones_por_1000_hab'] = base_enriquecida_comuna['publicaciones_totales'] / base_enriquecida_comuna['poblacion_total'] * 1000
base_enriquecida_comuna['publicaciones_por_km2'] = base_enriquecida_comuna['publicaciones_totales'] / base_enriquecida_comuna['superficie_km2']

display(base_enriquecida_comuna.sort_values('publicaciones_por_1000_hab', ascending=False))
base_enriquecida_comuna.to_csv(PROCESSED_DIR / 'eda_comunas_censo_oferta_v1.csv', index=False)

## Visualización inicial de oferta relativa

In [ ]:
try:
    import matplotlib.pyplot as plt
    plot_df = base_enriquecida_comuna.sort_values('publicaciones_por_1000_hab')
    plt.figure(figsize=(10, 7))
    plt.barh(plot_df['comuna'].astype(str), plot_df['publicaciones_por_1000_hab'])
    plt.xlabel('Publicaciones por 1000 habitantes')
    plt.ylabel('Comuna')
    plt.title('Oferta publicada relativa a poblacion por comuna')
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print(exc)

## Cruce por operación y comuna

In [ ]:
oferta_operacion_comuna = oferta_comuna.merge(censo_2022[['comuna', 'poblacion_total', 'superficie_km2']], on='comuna', how='left')
oferta_operacion_comuna['publicaciones_por_1000_hab'] = oferta_operacion_comuna['publicaciones'] / oferta_operacion_comuna['poblacion_total'] * 1000
display(oferta_operacion_comuna.sort_values(['tipo_operacion_norm', 'publicaciones_por_1000_hab'], ascending=[True, False]).head(60))
oferta_operacion_comuna.to_csv(PROCESSED_DIR / 'eda_operacion_comuna_censo_v1.csv', index=False)

## Cierre EDA con fuentes externas

Estos cruces son descriptivos y agregados a comuna. Sirven para detectar concentración relativa de oferta, pero no explican diferencias entre barrios dentro de una misma comuna. Para etapas posteriores, las fuentes puntuales de transporte, salud, educación y cultura deberían integrarse mediante distancia o asignación geográfica.